In [ ]:
# تثبيت الحزم المتوافقة مع كولاب
!pip install -q --no-deps "xformers<0.0.29" trl peft accelerate bitsandbytes
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q datasets huggingface_hub

In [ ]:
import torch

# التأكد من عمل كارت الشاشة GPU المخصص (T4 أو A100)
if torch.cuda.is_available():
    print(f"✅ GPU متوفر: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ تحذير: GPU غير متوفر! تأكد من تغيير Runtime Type إلى T4 GPU على Colab.")

In [ ]:
from datasets import load_dataset

print("⏳ جاري تحميل Spider dataset من Hugging Face...")
# تحميل النسخة الجاهزة من Spider
dataset = load_dataset("xlangai/spider", split="train")

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are a text-to-SQL system. Given the database schema context, convert the natural language question into a valid SQLite SQL query.

Database Schema:
{schema}

### Input:
{question}

### Response:
{query}"""

EOS_TOKEN = "<|endoftext|>"

def format_prompts(examples):
    schemas = examples["db_schema"]
    questions = examples["question"]
    queries = examples["query"]
    
    texts = []
    for schema, question, query in zip(schemas, questions, queries):
        text = alpaca_prompt.format(
            schema=str(schema) if schema else "",
            question=question,
            query=query
        ) + EOS_TOKEN
        texts.append(text)
        
    return {"text": texts}

formatted_dataset = dataset.map(format_prompts, batched=True)
print(f"✅ تم تجهيز وتنسيق البيانات! إجمالي العينات: {len(formatted_dataset)}")

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None
load_in_4bit = True

# 1. تحميل النموذج المكمم الأساسي
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# 2. إعداد مصفوفات LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print("✅ تم إعداد النموذج والطبقات بنجاح.")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=60,  # عدد الخطوات التجريبية
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

print("🚀 بدء عملية التدريب...")
trainer_stats = trainer.train()
print("🎉 اكتمل التدريب بنجاح!")

In [ ]:
from huggingface_hub import notebook_login

# 1. تسجيل الدخول إلى Hugging Face
notebook_login()

# 2. تحديد اسم الريبو على حسابك في Hugging Face
HF_REPO_NAME = "اسم_حسابك_هنا/Qwen2.5-Coder-7B-Text2SQL-LoRA"

# حفظ ورفع الـ LoRA Adapters فقط (حجمها خفيف حوالي 100-200 ميجابايت)
model.push_to_hub_merged(HF_REPO_NAME, tokenizer, save_method="lora")
print(f"✅ تم رفع الأوزان بنجاح إلى: https://huggingface.co/{HF_REPO_NAME}")

In [ ]:
git add notebooks/02_fine_tuning_unsloth.ipynb
git commit -m "feat: add unsloth fine-tuning notebook for qwen2.5-coder"
git push origin Abdallah_branch